# C3 · extracción óptima — notebook de análisis (`debug`)

**Objeto:** ROXs42Bb  |  **Run:** `ROXs42Bb_realigned`  |  **Spec:** [`docs/spec_C3_codex_optimal_extraction.md`](../../../docs/spec_C3_codex_optimal_extraction.md)

Rehace C3 **dentro del notebook**, con el código a la vista y editable, para probar y ajustar sin tocar `musepipe`. El notebook de auditoría es [`../C3_optimal.ipynb`](../C3_optimal.ipynb).

**C3 produce dos métodos, no uno.** El estimador es el mismo — Horne (1986): por canal, cada píxel pesa por el perfil de PSF esperado y por la inversa de su varianza, `f = Σ M·P·D/V ÷ Σ M·P²/V` — y lo que cambia es **el cubo del que se extrae**:

| variante | cubo | por qué existe |
|---|---|---|
| `optimal_ls` | residual de superficie local (04b) | **mismo fondo que C2**, así que compararlos aísla la ganancia del ponderado óptimo |
| `optimal_psfsub` | cubo de B2 menos el **modelo de PSF de la primaria**, ajustado aquí canal a canal | anticipa el fondo de C4; `ls` vs `psfsub` es el diagnóstico del modelo de halo que consume D1 |

Aquí se hacen **las dos**, en paralelo, y la comparación final las contrasta por separado contra sus productos de la cadena.


In [ ]:
import json, sys
from pathlib import Path

import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt

# Resolución de las figuras EN PANTALLA. `savefig` guarda a 300 dpi, pero
# lo que se ve dentro del notebook lo fija el backend inline, que va a 100
# dpi por defecto y sale borroso. `retina` dobla los píxeles sin cambiar el
# tamaño aparente; fuera de IPython no hace nada y queda el rcParam.
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 120
mpl.rcParams['savefig.dpi'] = 200
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats('retina')
except Exception:
    pass
_here = Path.cwd()
ROOT = next(p for p in (_here, *_here.parents) if (p / 'musepipe').is_dir())
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / 'notebooks'))
import _nbcommon as nb

RUN_ID = nb.resolve_run_id('ROXs42Bb_realigned')
RD = nb.run_dir(RUN_ID); SD = RD / 'stages'
CFG = json.loads((RD / 'config' / 'config.json').read_text(encoding='utf-8'))['config']
TARGET = nb.run_target(RUN_ID) or nb.display_name(RUN_ID)
print('objeto :', TARGET, '·', nb.display_name(RUN_ID))
print('run    :', RUN_ID)


## 1 · Perillas

Salen del **config resuelto de la etapa**, no del `config.json` crudo: C3 rellena defaults que no están escritos en el run (`x02_local_bkg_annulus_px` hereda de `x01_annulus_bkg_px`, el radio de ajuste de la primaria de `psf_norm_radius_px`…), y copiarlos a mano es exactamente cómo se consigue un notebook que no reproduce la cadena. Cambia lo que quieras **debajo** de la lectura y vuelve a ejecutar.

`WINDOW_RADIUS_PX` es la que más mueve el resultado: define hasta dónde llega el ponderado, y la fracción de PSF que queda fuera la recupera después `apcorr`.


In [ ]:
from musepipe.stages.stage_x02_optimal import stage_x02_config_from_run

# `project_root=ROOT` no es opcional: musepipe resuelve rutas contra el cwd, y
# el cwd de un notebook es su propia carpeta, no la raíz del repo.
X02 = stage_x02_config_from_run(RUN_ID, project_root=ROOT)   # run + defaults de la etapa
WINDOW_RADIUS_PX     = float(X02.get('x02_window_radius_px', 8.0))
CLIP_SIGMA           = float(X02.get('x02_clip_sigma', 4.0))
CLIP_MAX_ITER        = int(X02.get('x02_clip_max_iter', 2))
APCORR_MODE          = X02.get('x02_aperture_correction', 'auto')
ERROR_MODE           = X02.get('x02_error_mode', 'auto')
N_CONTROLS           = int(X02.get('x02_control_apertures', 8))
EXCLUDE_ANGLE_DEG    = float(X02.get('x02_control_exclude_angle_deg', 25.0))
LOCAL_BKG_ANNULUS_PX = X02.get('x02_local_bkg_annulus_px')
PRIMARY_FIT_RADIUS   = float(X02.get('x02_primary_fit_radius_px', 25.0))
PRIMARY_EXCL_RADIUS  = float(X02.get('x02_primary_exclude_radius_px', WINDOW_RADIUS_PX))
BAD_WINDOWS_A        = X02.get('x02_bad_windows_A', [])
SKYLINE_WINDOWS_A    = X02.get('x02_skyline_windows_A', [])
INTERPOLATED_WIN_A   = X02.get('x02_interpolated_windows_A', [])

# ---- a partir de aquí, cambia lo que quieras probar ----

for _k, _v in sorted({'ventana (px)': WINDOW_RADIUS_PX, 'clip σ': CLIP_SIGMA,
                      'clip iter': CLIP_MAX_ITER, 'apcorr': APCORR_MODE,
                      'modo error': ERROR_MODE, 'controles': N_CONTROLS,
                      'anillo fondo': LOCAL_BKG_ANNULUS_PX,
                      'radio ajuste primaria': PRIMARY_FIT_RADIUS,
                      'radio exclusión compañero': PRIMARY_EXCL_RADIUS}.items()):
    print(f'  {_k:26s} {_v}')


## 2 · Entradas — **los dos cubos**

`ls` sale del residual de 04b; `psfsub` del cubo de B2. Los dos deben tener la misma forma: la cadena lo exige y para aquí si no (serían dos rejillas distintas).


In [ ]:
qc_b3 = json.loads((SD / 'stage01c_qc.json').read_text(encoding='utf-8'))
OBJECT_YX = tuple(float(v) for v in qc_b3['companion']['pos_yx'])
STAR_YX   = tuple(float(v) for v in qc_b3['primary']['pos_yx'])
PSF_MODEL = json.loads((SD / 'psf_model.json').read_text(encoding='utf-8'))

CUBE_PATH = SD / 'stage02_xcorr_cube_stack.fits'
with fits.open(CUBE_PATH) as h:
    STAGE02 = np.asarray(h['CUBES'].data, dtype=float)
    WAVE = np.asarray(h['WAVELENGTH'].data, dtype=float)
    STAT_CUBE = np.asarray(h['STAT'].data, dtype=float) if 'STAT' in h else None
    _stack_bunit = str(h[0].header.get('BUNIT', '')
                       or h['CUBES'].header.get('BUNIT', '')) or None
# La unidad, con la MISMA regla que la cadena: este stack no lleva BUNIT
# (se escribió antes de que B1/B2 lo propagaran), así que `resolve_bunit`
# cae al cubo de entrada del run. Sin esto las colorbars no tienen unidad.
from musepipe.io import resolve_bunit
BUNIT = resolve_bunit(X02, stack_bunit=_stack_bunit)
UNIDAD = BUNIT or 'sin unidad declarada'
if STAGE02.ndim == 4:
    STAGE02 = STAGE02[0]
if STAT_CUBE is not None and STAT_CUBE.ndim == 4:
    STAT_CUBE = STAT_CUBE[0]
LS_04B = np.asarray(fits.getdata(SD / 'cube_residual_local_object.fits'), dtype=float)
assert LS_04B.shape == STAGE02.shape, (LS_04B.shape, STAGE02.shape)
# Wings-intact (2026-07-26): con anillo configurado, LS extrae del cubo CRUDO,
# igual que C2. Antes salia del residual de 04b y encima se le restaba el
# anillo: dos fondos sobre un cubo que no es homogeneo (04b solo resta
# alrededor del objeto). Ver reports/20260726/auditoria_c3_doble_sustraccion.
WINGS_INTACT_LS = bool(X02.get('x02_wings_intact_ls', True))
LS_CUBE = (STAGE02 if (WINGS_INTACT_LS and LOCAL_BKG_ANNULUS_PX is not None)
           else LS_04B)
print('cubo de ls:', 'crudo de B2 (wings-intact)' if LS_CUBE is STAGE02
      else 'residual de 04b (historico)')

qc00 = json.loads((SD / 'stage00q_qc.json').read_text(encoding='utf-8'))
qc01 = json.loads((SD / 'stage01_qc.json').read_text(encoding='utf-8'))
m5 = qc00.get('m5_stat', {})
# El STAT crudo se multiplica por el factor POR SPAXEL de M5 (aquí no es 1:
# el DRS subestima la varianza) antes de entrar como peso del estimador.
STAT_FACTOR = float(X02.get('x02_stat_factor_spaxel',
                            m5.get('factor_spaxel_median', 1.0)) or 1.0)
COV_FACTOR  = float(X02.get('x02_covariance_factor_box3',
                            qc01.get('stat', {}).get('covariance_factor_box3', 1.0)) or 1.0)
STAT_STATUS = str(X02.get('x02_stat_status', m5.get('status', 'unknown')))
print('cubos    :', STAGE02.shape, '| STAT:', 'sí' if STAT_CUBE is not None else 'no')
print('compañero:', [round(v, 2) for v in OBJECT_YX], ' primaria:', [round(v, 2) for v in STAR_YX])
print(f'STAT     : factor={STAT_FACTOR:.3f} covarianza={COV_FACTOR:.3f} estado={STAT_STATUS}')
print('BUNIT    :', UNIDAD)


## 3 · De dónde sale el espectro

Antes de nada, **dónde** se mide. A diferencia de C2, que suma una caja 3×3, aquí la extracción usa una **ventana circular** de `WINDOW_RADIUS_PX` px alrededor del compañero: no es una apertura que se sume: es el conjunto de píxeles que entran en el estimador, cada uno con su peso.

Las dos imágenes son el **mismo** cubo (la mediana en λ) con **dos escalas**, porque una sola no puede enseñar las dos cosas:

- **logarítmica**: enseña el halo AO de la primaria, que es el fondo contra el que hay que medir y lo que las dos variantes tratan de forma distinta;
- **lineal recortada al entorno del compañero**: enseña la fuente, que en log queda aplastada contra el halo.

Marcado encima: la **ventana de extracción** (círculo continuo), el **anillo de fondo** cuando está configurado (dos círculos punteados) con el **disco excluido** alrededor de la primaria, y las dos posiciones que vienen de B3.


In [ ]:
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle

# La mediana se toma en la BANDA ROJA, no en todo el rango: es donde el
# compañero se detecta. Promediando los 3681 canales manda el azul, que
# es casi todo halo y ruido, y la fuente no asoma ni con el halo quitado.
BANDA_IMAGEN = (7500.0, 9000.0)     # cámbiala y vuelve a ejecutar
_ch = (WAVE >= BANDA_IMAGEN[0]) & (WAVE <= BANDA_IMAGEN[1])
img_med = np.nanmedian(LS_CUBE[_ch], axis=0)   # el cubo del que extrae `ls`
# El recorte enmarca a las DOS fuentes: el disco excluido alrededor de
# la primaria es parte de lo que se está explicando, y con un recorte
# centrado en el compañero se dibujaba fuera de la imagen.
MARGEN_PX = 16.0
_ny, _nx = img_med.shape
sl = (slice(int(max(0, min(STAR_YX[0], OBJECT_YX[0]) - MARGEN_PX)),
            int(min(_ny, max(STAR_YX[0], OBJECT_YX[0]) + MARGEN_PX + 1))),
      slice(int(max(0, min(STAR_YX[1], OBJECT_YX[1]) - MARGEN_PX)),
            int(min(_nx, max(STAR_YX[1], OBJECT_YX[1]) + MARGEN_PX + 1))))
y0, x0 = sl[0].start, sl[1].start
rec = img_med[sl]

def _marcas(ax):
    ax.add_patch(Circle((OBJECT_YX[1] - x0, OBJECT_YX[0] - y0), WINDOW_RADIUS_PX,
                        fill=False, color='tab:cyan', lw=1.4))
    if LOCAL_BKG_ANNULUS_PX is not None:
        for r in LOCAL_BKG_ANNULUS_PX[:2]:
            ax.add_patch(Circle((OBJECT_YX[1] - x0, OBJECT_YX[0] - y0), float(r),
                                fill=False, color='tab:orange', lw=0.9, ls=':'))
        r_ex = float(LOCAL_BKG_ANNULUS_PX[2]) if len(LOCAL_BKG_ANNULUS_PX) > 2 else 30.0
        ax.add_patch(Circle((STAR_YX[1] - x0, STAR_YX[0] - y0), r_ex,
                            fill=False, color='tab:red', lw=0.9, ls='--'))
    ax.plot(STAR_YX[1] - x0, STAR_YX[0] - y0, '*', color='w', ms=11, mec='k')
    ax.plot(OBJECT_YX[1] - x0, OBJECT_YX[0] - y0, '+', color='tab:cyan', ms=9)

# Tercera vista, SOLO PARA VER: se le quita al campo el perfil radial
# mediano alrededor de la primaria. El halo AO es casi simétrico en
# azimut, así que al restarlo lo que sobresale es el compañero. No
# entra en ninguna cuenta — es lo que hacen C5/C6, no C3.
yy_f, xx_f = np.indices(img_med.shape, dtype=float)
r_bin = np.hypot(yy_f - STAR_YX[0], xx_f - STAR_YX[1]).astype(int)
perfil = np.full(int(r_bin.max()) + 1, np.nan)
for b in range(perfil.size):
    m_ = r_bin == b
    if m_.any():
        perfil[b] = np.nanmedian(img_med[m_])
sin_halo = (img_med - perfil[r_bin])[sl]

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(15, 5.0))
# Escala LOG sobre los positivos: el halo abarca varios órdenes de magnitud.
pos = rec[np.isfinite(rec) & (rec > 0)]
vmin = float(np.nanpercentile(pos, 30)) if pos.size else 1e-3
vmax = float(np.nanpercentile(pos, 99.9)) if pos.size else 1.0
im1 = a1.imshow(rec, origin='lower', cmap='magma',
                norm=LogNorm(vmin=max(vmin, 1e-6), vmax=max(vmax, vmin * 10)))
a1.set_title(f'mediana {BANDA_IMAGEN[0]:.0f}-{BANDA_IMAGEN[1]:.0f} Å · LOG (el halo de la primaria)', fontsize=9)
cb1 = fig.colorbar(im1, ax=a1, fraction=0.046)
cb1.set_label(f'flujo mediano [{UNIDAD}]', fontsize=7)
# Escala LINEAL acotada por lo que hay CERCA del compañero, no por la
# primaria: si no, el compañero es un píxel indistinguible del fondo.
yy_, xx_ = np.indices(rec.shape, dtype=float)
cerca = np.hypot(yy_ - (OBJECT_YX[0] - y0), xx_ - (OBJECT_YX[1] - x0)) <= 2 * WINDOW_RADIUS_PX
lo, hi = np.nanpercentile(rec[cerca & np.isfinite(rec)], [5, 99.5])
im2 = a2.imshow(rec, origin='lower', cmap='viridis', vmin=lo, vmax=hi)
a2.set_title('zoom al compañero · escala LINEAL a su entorno', fontsize=9)
cb2 = fig.colorbar(im2, ax=a2, fraction=0.046)
cb2.set_label(f'flujo mediano [{UNIDAD}]', fontsize=7)
v3 = float(np.nanpercentile(np.abs(sin_halo[cerca]), 98))
im3 = a3.imshow(sin_halo, origin='lower', cmap='RdBu_r', vmin=-v3, vmax=v3)
a3.set_title('zoom · perfil radial del halo restado (solo para ver)', fontsize=9)
cb3 = fig.colorbar(im3, ax=a3, fraction=0.046, extend='both')
cb3.set_label(f'flujo − halo azimutal [{UNIDAD}]', fontsize=7)
for ax in (a1, a2, a3):
    _marcas(ax)
    # Que un círculo grande no estire los ejes y encoja la imagen.
    ax.set_xlim(-0.5, rec.shape[1] - 0.5); ax.set_ylim(-0.5, rec.shape[0] - 0.5)
    ax.set_xlabel('x [px]'); ax.set_ylabel('y [px]')
# El panel derecho hace zoom: a esta separación el compañero ocupa unos
# pocos píxeles y en el campo entero no se distingue del fondo.
_zoom = 2.6 * float(WINDOW_RADIUS_PX)
for ax in (a2, a3):
    ax.set_xlim(OBJECT_YX[1] - x0 - _zoom, OBJECT_YX[1] - x0 + _zoom)
    ax.set_ylim(OBJECT_YX[0] - y0 - _zoom, OBJECT_YX[0] - y0 + _zoom)
# Cuánto sobresale el compañero una vez quitado el halo azimutal.
_en_win = (np.hypot(yy_ - (OBJECT_YX[0] - y0), xx_ - (OBJECT_YX[1] - x0))
           <= float(WINDOW_RADIUS_PX))
print(f'con el halo azimutal quitado, en la ventana: mediana'
      f' {float(np.nanmedian(sin_halo[_en_win])):7.2f}, pico'
      f' {float(np.nanmax(sin_halo[_en_win])):7.2f} {UNIDAD}')
fig.suptitle('dónde se extrae: ventana (cian), anillo de fondo (naranja punteado),'
             ' disco excluido de la primaria (rojo)', fontsize=9)
fig.tight_layout(); plt.show()

# El mismo criterio que `circular_window_indices` (que se copia más
# abajo): distancia al centro <= radio. Aquí a mano para no depender
# de una celda posterior.
yy_g, xx_g = np.indices(LS_CUBE.shape[1:], dtype=float)
n_win = int((np.hypot(yy_g - OBJECT_YX[0], xx_g - OBJECT_YX[1])
             <= float(WINDOW_RADIUS_PX)).sum())
print(f'ventana de extracción: círculo de r={float(WINDOW_RADIUS_PX):g} px'
      f' -> {n_win} píxeles (una caja 3×3 tendría 9)')
sep = float(np.hypot(OBJECT_YX[0] - STAR_YX[0], OBJECT_YX[1] - STAR_YX[1]))
print(f'separación compañero-primaria: {sep:.1f} px')
if LOCAL_BKG_ANNULUS_PX is not None:
    print(f'anillo de fondo: {LOCAL_BKG_ANNULUS_PX[0]:g}-{LOCAL_BKG_ANNULUS_PX[1]:g} px'
          f' | excluye r<{(LOCAL_BKG_ANNULUS_PX[2] if len(LOCAL_BKG_ANNULUS_PX) > 2 else 30.0):g} px'
          ' de la primaria')
else:
    print('sin anillo de fondo configurado: `ls` extrae del residual de 04b')


## 4 · Las funciones numéricas, copiadas de `musepipe`

Copia **literal**; edítalas y el resultado cambia. Se importan solo `evaluate_psf_model` (es de C1) y `run_channel_chunks` (paralelismo, no física).

- `finite_values` — de `musepipe/stats.py`
- `robust_sigma` — de `musepipe/stats.py`
- `robust_sigma_axis0` — de `musepipe/stats.py`
- `angular_separation_deg` — de `musepipe/apertures.py`
- `aperture_weights` — de `musepipe/apertures.py`
- `same_radius_control_positions` — de `musepipe/apertures.py`
- `_as_cube` — de `musepipe/extraction/aperture.py`
- `annulus_background_spectrum` — de `musepipe/extraction/aperture.py`
- `_flag_window` — de `musepipe/extraction/aperture.py`
- `channel_flags` — de `musepipe/extraction/aperture.py`
- `aperture_correction_from_psf` — de `musepipe/extraction/aperture.py`
- `circular_window_indices` — de `musepipe/extraction/optimal.py`
- `normalized_psf_window` — de `musepipe/extraction/optimal.py`
- `covariance_factor_for_npix` — de `musepipe/extraction/optimal.py`
- `_channel_estimate` — de `musepipe/extraction/optimal.py`
- `estimate_variance_cube` — de `musepipe/extraction/optimal.py`
- `optimal_raw_spectrum` — de `musepipe/extraction/optimal.py`
- `control_optimal_spectra` — de `musepipe/extraction/optimal.py`
- `psf_image` — de `musepipe/extraction/optimal.py`
- `fit_primary_psf_model_cube` — de `musepipe/extraction/optimal.py`


In [ ]:
# ------------------------------------------------------------------
# COPIA EDITABLE. Fuente: musepipe (ver el chequeo de deriva abajo).
# ------------------------------------------------------------------
from musepipe.parallel import run_channel_chunks
from musepipe.psf import evaluate_psf_model
from typing import Sequence
import math
import numpy as np
import warnings
# `evaluate_psf_model` (C1) y `run_channel_chunks` (paralelismo) se importan
# arriba: no son lo que se ajusta aquí.

FLAG_BAD_WINDOW = 1
FLAG_SKYLINE = 2
FLAG_INTERPOLATED = 4
FLAG_CLIPPED = 8


def finite_values(values) -> np.ndarray:
    """Return finite values as a float64 1D array."""

    arr = np.asarray(values, dtype=np.float64)
    return arr[np.isfinite(arr)]


def robust_sigma(values) -> float:
    """Robust 1D sigma estimate using MAD with std fallback."""

    vals = finite_values(values)
    if vals.size == 0:
        return np.nan
    med = np.nanmedian(vals)
    mad = np.nanmedian(np.abs(vals - med))
    sigma = 1.4826 * mad
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = np.nanstd(vals)
    return float(sigma)


def robust_sigma_axis0(values) -> np.ndarray:
    """Robust sigma along axis 0 using MAD with std fallback per column."""

    arr = np.asarray(values, dtype=np.float64)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        med = np.nanmedian(arr, axis=0)
        mad = np.nanmedian(np.abs(arr - med[None, :]), axis=0)
        sigma = 1.4826 * mad
        std = np.nanstd(arr, axis=0)
    bad = ~np.isfinite(sigma) | (sigma <= 0)
    sigma[bad] = std[bad]
    return sigma


def angular_separation_deg(a, b) -> float:
    """Smallest angular separation between two angles in radians, in degrees."""

    return abs(math.degrees(math.atan2(math.sin(a - b), math.cos(a - b))))


def aperture_weights(ny, nx, center_yx, aperture) -> np.ndarray:
    """Build a 2D aperture-weight image."""

    y0, x0 = map(float, center_yx)
    yy, xx = np.mgrid[:ny, :nx]
    rr2 = (yy - y0) ** 2 + (xx - x0) ** 2
    weights = np.zeros((ny, nx), dtype=np.float64)
    kind = aperture["kind"]

    if kind == "pixel":
        y = int(round(y0))
        x = int(round(x0))
        if 0 <= y < ny and 0 <= x < nx:
            weights[y, x] = 1.0
    elif kind == "box":
        size = int(aperture.get("size", 3))
        half = size // 2
        y = int(round(y0))
        x = int(round(x0))
        y1 = max(0, y - half)
        y2 = min(ny, y + half + 1)
        x1 = max(0, x - half)
        x2 = min(nx, x + half + 1)
        weights[y1:y2, x1:x2] = 1.0
    elif kind == "circle":
        radius = float(aperture["radius_px"])
        weights[rr2 <= radius**2] = 1.0
    elif kind == "gaussian":
        sigma = float(aperture["sigma_px"])
        radius = float(aperture.get("radius_px", 3.0 * sigma))
        mask = rr2 <= radius**2
        weights[mask] = np.exp(-0.5 * rr2[mask] / sigma**2)
    else:
        raise ValueError(f"Unknown aperture kind: {kind}")
    return weights


def same_radius_control_positions(
    object_yx,
    star_yx,
    ny,
    nx,
    n_positions=8,
    exclude_angle_deg=25.0,
    margin_px=4,
):
    """Return integer control positions at the same star-object radius."""

    oy, ox = map(float, object_yx)
    sy, sx = map(float, star_yx)
    dy = oy - sy
    dx = ox - sx
    radius = math.hypot(dy, dx)
    theta0 = math.atan2(dy, dx)

    controls = []
    for k in range(int(n_positions)):
        theta = theta0 + 2.0 * math.pi * k / float(n_positions)
        if angular_separation_deg(theta, theta0) < exclude_angle_deg:
            continue
        y = int(round(sy + radius * math.sin(theta)))
        x = int(round(sx + radius * math.cos(theta)))
        if margin_px <= y < ny - margin_px and margin_px <= x < nx - margin_px:
            controls.append((y, x))
    return controls


def _as_cube(cube_zyx, name="cube") -> np.ndarray:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected {name} with shape (nz,ny,nx), got {cube.shape}.")
    return cube


def annulus_background_spectrum(cube_zyx, center_yx, r_in, r_out, *, exclude_yx=None, exclude_radius=0.0):
    """Per-channel local background = median of a source-free annulus.

    Used for the wings-intact aperture-correction path: subtracting a distant
    annulus (rather than a local surface, stage04b) preserves the companion's
    PSF wings so the PSF growth-curve aperture correction stays self-consistent
    (box3<box5). Excludes a region around ``exclude_yx`` (the primary)."""

    cube = _as_cube(cube_zyx)
    _, ny, nx = cube.shape
    yy, xx = np.mgrid[0:ny, 0:nx]
    r = np.hypot(yy - float(center_yx[0]), xx - float(center_yx[1]))
    mask = (r >= float(r_in)) & (r <= float(r_out))
    if exclude_yx is not None and float(exclude_radius) > 0:
        mask &= np.hypot(yy - float(exclude_yx[0]), xx - float(exclude_yx[1])) > float(exclude_radius)
    if not mask.any():
        return np.zeros(cube.shape[0], dtype=np.float64)
    vals = cube[:, mask]
    with np.errstate(all="ignore"):
        return np.nanmedian(vals, axis=1).astype(np.float64)


def _flag_window(wave_A: np.ndarray, windows_A: Sequence[Sequence[float]], bit: int, flags: np.ndarray) -> None:
    for window in windows_A or ():
        if window is None or len(window) != 2:
            continue
        lo, hi = window
        if lo is None or hi is None:
            continue
        flags[(wave_A >= float(lo)) & (wave_A <= float(hi))] |= int(bit)


def channel_flags(
    wave_A,
    *,
    bad_windows_A: Sequence[Sequence[float]] = (),
    skyline_windows_A: Sequence[Sequence[float]] = (),
    interpolated_windows_A: Sequence[Sequence[float]] = (),
    clipped_mask=None,
    good_mask=None,
    bad_mask=None,
) -> np.ndarray:
    wave = np.asarray(wave_A, dtype=np.float64)
    flags = np.zeros(wave.size, dtype=np.int32)
    _flag_window(wave, bad_windows_A, FLAG_BAD_WINDOW, flags)
    _flag_window(wave, skyline_windows_A, FLAG_SKYLINE, flags)
    _flag_window(wave, interpolated_windows_A, FLAG_INTERPOLATED, flags)
    if good_mask is not None:
        flags[~np.asarray(good_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if bad_mask is not None:
        flags[np.asarray(bad_mask, dtype=bool)] |= FLAG_BAD_WINDOW
    if clipped_mask is not None:
        flags[np.asarray(clipped_mask, dtype=bool)] |= FLAG_CLIPPED
    return flags


def aperture_correction_from_psf(
    wave_A,
    aperture: dict,
    psf_model: dict | None,
    *,
    center_yx=(0.0, 0.0),
    correction_mode: str = "auto",
) -> tuple[np.ndarray, str, float]:
    """Return wavelength-dependent aperture correction from a C1 PSF model."""

    wave = np.asarray(wave_A, dtype=np.float64)
    mode = str(correction_mode or "auto").lower()
    if mode in {"none", "off", "false"}:
        return np.ones(wave.size, dtype=np.float64), "none", 0.0
    if psf_model is None:
        if mode in {"auto", "optional"}:
            return np.ones(wave.size, dtype=np.float64), "none", 0.0
        raise RuntimeError("Aperture correction requested but no psf_model was supplied.")

    norm_radius = float(psf_model.get("norm_radius_px", 25.0))
    half = int(math.ceil(norm_radius))
    frac_y = float(center_yx[0]) - round(float(center_yx[0]))
    frac_x = float(center_yx[1]) - round(float(center_yx[1]))
    source_center = (half + frac_y, half + frac_x)
    yy, xx = np.indices((2 * half + 1, 2 * half + 1), dtype=np.float64)
    dy = yy - source_center[0]
    dx = xx - source_center[1]
    weights = aperture_weights(2 * half + 1, 2 * half + 1, source_center, aperture)
    fractions = np.empty(wave.size, dtype=np.float64)
    for i, w in enumerate(wave):
        psf = evaluate_psf_model(psf_model, float(w), dy, dx)
        frac = float(np.nansum(psf * weights))
        if not np.isfinite(frac) or frac <= 0:
            raise RuntimeError(f"Invalid aperture PSF fraction at wave={w}.")
        fractions[i] = frac
    return (1.0 / fractions).astype(np.float64), "psf_growth_curve", norm_radius


def circular_window_indices(shape, center_yx, radius_px):
    ny, nx = map(int, shape)
    cy, cx = map(float, center_yx)
    radius = float(radius_px)
    half = int(math.ceil(radius))
    y1 = max(0, int(math.floor(cy)) - half)
    y2 = min(ny, int(math.floor(cy)) + half + 2)
    x1 = max(0, int(math.floor(cx)) - half)
    x2 = min(nx, int(math.floor(cx)) + half + 2)
    yy, xx = np.mgrid[y1:y2, x1:x2]
    mask = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius**2
    return yy[mask].astype(int), xx[mask].astype(int)


def normalized_psf_window(psf_model, wavelength_A, center_yx, ypix, xpix):
    p = evaluate_psf_model(
        psf_model,
        float(wavelength_A),
        np.asarray(ypix, dtype=np.float64) - float(center_yx[0]),
        np.asarray(xpix, dtype=np.float64) - float(center_yx[1]),
    ).astype(np.float64)
    p[~np.isfinite(p)] = 0.0
    p[p < 0] = 0.0
    norm = float(np.sum(p))
    if not np.isfinite(norm) or norm <= 0:
        raise RuntimeError("PSF window has invalid normalization.")
    return p / norm


def covariance_factor_for_npix(npix_eff, covariance_factor_box3=1.0):
    """Linearly interpolate covariance inflation between one pixel and box3."""

    vals = np.asarray(npix_eff, dtype=np.float64)
    box3 = float(covariance_factor_box3)
    if not np.isfinite(box3) or box3 <= 0:
        box3 = 1.0
    t = np.clip((vals - 1.0) / 8.0, 0.0, 1.0)
    return 1.0 + t * (box3 - 1.0)


def _channel_estimate(data, variance, p, valid, *, clip_sigma=4.0, clip_max_iter=2):
    data = np.asarray(data, dtype=np.float64)
    variance = np.asarray(variance, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    valid = np.asarray(valid, dtype=bool)
    mask = valid.copy()
    n_valid = int(np.count_nonzero(valid))
    if n_valid == 0:
        return np.nan, np.nan, np.nan, 1.0, np.zeros_like(valid, dtype=bool)

    flux = np.nan
    raw_var = np.nan
    max_iter = max(0, int(clip_max_iter))
    for iteration in range(max_iter + 1):
        denom = np.sum((p[mask] ** 2) / variance[mask])
        if not np.isfinite(denom) or denom <= 0:
            return np.nan, np.nan, np.nan, 1.0, valid.copy()
        flux = float(np.sum(p[mask] * data[mask] / variance[mask]) / denom)
        raw_var = float(1.0 / denom)
        if clip_sigma is None or iteration >= max_iter:
            break
        z = np.zeros_like(data, dtype=np.float64)
        z[valid] = (data[valid] - flux * p[valid]) / np.sqrt(variance[valid])
        new_mask = valid & (np.abs(z) <= float(clip_sigma))
        if np.array_equal(new_mask, mask):
            break
        mask = new_mask

    clipped = valid & ~mask
    p_kept = p[mask]
    npix_eff = np.nan
    if p_kept.size and np.sum(p_kept**2) > 0:
        npix_eff = float((np.sum(p_kept) ** 2) / np.sum(p_kept**2))
    frac_clip = float(np.count_nonzero(clipped) / max(n_valid, 1))
    return flux, raw_var, npix_eff, frac_clip, clipped


def estimate_variance_cube(cube_zyx):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    out = np.empty_like(cube, dtype=np.float64)
    for i in range(cube.shape[0]):
        sigma = robust_sigma(cube[i])
        if not np.isfinite(sigma) or sigma <= 0:
            sigma = 1.0
        out[i] = sigma**2
    return out


def optimal_raw_spectrum(
    cube_zyx,
    variance_zyx,
    wave_A,
    center_yx,
    psf_model,
    *,
    window_radius_px=8.0,
    clip_sigma=4.0,
    clip_max_iter=2,
    n_jobs=1,
    bkg_spectrum=None,
):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    variance = np.asarray(variance_zyx, dtype=np.float64)
    wave = np.asarray(wave_A, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    if variance.shape != cube.shape:
        raise ValueError("variance_zyx shape must match cube_zyx.")
    if wave.ndim != 1 or wave.size != cube.shape[0]:
        raise ValueError("wave_A must be 1D and match cube spectral length.")
    if bkg_spectrum is not None:
        bkg_spectrum = np.asarray(bkg_spectrum, dtype=np.float64)
        if bkg_spectrum.shape != wave.shape:
            raise ValueError("bkg_spectrum must match wave_A length.")

    nz, ny, nx = cube.shape
    ypix, xpix = circular_window_indices((ny, nx), center_yx, window_radius_px)
    flux = np.full(nz, np.nan, dtype=np.float64)
    raw_var = np.full(nz, np.nan, dtype=np.float64)
    npix_eff = np.full(nz, np.nan, dtype=np.float64)
    clip_fraction = np.zeros(nz, dtype=np.float64)
    chunk_rejection = {}

    def _estimate_range(z0, z1):
        # Per-channel work identical to the serial loop; the rejection counts
        # accumulate in a per-chunk map (integer-valued, so the final sum is
        # exact regardless of chunk order).
        local_map = np.zeros((ny, nx), dtype=np.float64)
        for z in range(z0, z1):
            data = cube[z, ypix, xpix]
            if bkg_spectrum is not None and np.isfinite(bkg_spectrum[z]):
                # Local background reference: subtracting a per-channel scalar
                # is separable from the Horne estimator (D1 v2 §3.1).
                data = data - bkg_spectrum[z]
            var = variance[z, ypix, xpix]
            p = normalized_psf_window(psf_model, wave[z], center_yx, ypix, xpix)
            valid = np.isfinite(data) & np.isfinite(var) & (var > 0) & np.isfinite(p) & (p > 0)
            f, v, neff, frac, clipped = _channel_estimate(
                data,
                var,
                p,
                valid,
                clip_sigma=clip_sigma,
                clip_max_iter=clip_max_iter,
            )
            flux[z] = f
            raw_var[z] = v
            npix_eff[z] = neff
            clip_fraction[z] = frac
            if np.any(clipped):
                np.add.at(local_map, (ypix[clipped], xpix[clipped]), 1.0)
        chunk_rejection[z0] = local_map

    run_channel_chunks(_estimate_range, nz, n_jobs=n_jobs)
    rejection_map = np.zeros((ny, nx), dtype=np.float64)
    for z0 in sorted(chunk_rejection):
        rejection_map += chunk_rejection[z0]

    return {
        "flux": flux,
        "variance": raw_var,
        "npix_eff": npix_eff,
        "clip_fraction": clip_fraction,
        "rejection_map": rejection_map,
    }


def control_optimal_spectra(
    cube_zyx,
    variance_zyx,
    wave_A,
    object_yx,
    star_yx,
    psf_model,
    *,
    window_radius_px=8.0,
    clip_sigma=4.0,
    clip_max_iter=2,
    n_controls=8,
    exclude_angle_deg=25.0,
    n_jobs=1,
    local_bkg_annulus_px=None,
):
    cube = np.asarray(cube_zyx, dtype=np.float64)
    _, ny, nx = cube.shape
    controls = same_radius_control_positions(
        object_yx,
        star_yx,
        ny,
        nx,
        n_positions=int(n_controls),
        exclude_angle_deg=float(exclude_angle_deg),
        margin_px=int(math.ceil(window_radius_px)) + 1,
    )
    spectra = []
    for center in controls:
        bkg = None
        if local_bkg_annulus_px is not None:
            bkg = annulus_background_spectrum(
                cube,
                center,
                local_bkg_annulus_px[0],
                local_bkg_annulus_px[1],
                exclude_yx=star_yx,
                exclude_radius=float(local_bkg_annulus_px[2]) if len(local_bkg_annulus_px) > 2 else 30.0,
            )
        raw = optimal_raw_spectrum(
            cube,
            variance_zyx,
            wave_A,
            center,
            psf_model,
            window_radius_px=window_radius_px,
            clip_sigma=clip_sigma,
            clip_max_iter=clip_max_iter,
            n_jobs=n_jobs,
            bkg_spectrum=bkg,
        )
        spectra.append(raw["flux"])
    if not spectra:
        return controls, np.empty((0, cube.shape[0]), dtype=np.float64)
    return controls, np.asarray(spectra, dtype=np.float64)


def psf_image(shape, wavelength_A, center_yx, psf_model):
    yy, xx = np.indices(shape, dtype=np.float64)
    return evaluate_psf_model(
        psf_model,
        float(wavelength_A),
        yy - float(center_yx[0]),
        xx - float(center_yx[1]),
    )


def fit_primary_psf_model_cube(
    cube_zyx,
    wave_A,
    primary_yx,
    psf_model,
    *,
    variance_zyx=None,
    fit_radius_px: float | None = None,
    exclude_centers_yx=(),
    exclude_radius_px: float = 8.0,
    n_jobs: int = 1,
) -> tuple[np.ndarray, dict]:
    cube = np.asarray(cube_zyx, dtype=np.float64)
    wave = np.asarray(wave_A, dtype=np.float64)
    if cube.ndim != 3:
        raise ValueError(f"Expected cube shape (nz,ny,nx), got {cube.shape}.")
    if wave.size != cube.shape[0]:
        raise ValueError("wave_A must match cube spectral length.")
    variance = None if variance_zyx is None else np.asarray(variance_zyx, dtype=np.float64)
    if variance is not None and variance.shape != cube.shape:
        raise ValueError("variance_zyx shape must match cube_zyx.")

    nz, ny, nx = cube.shape
    yy, xx = np.indices((ny, nx), dtype=np.float64)
    py, px = map(float, primary_yx)
    fit_radius = float(fit_radius_px or psf_model.get("norm_radius_px", 25.0))
    fit_mask = (yy - py) ** 2 + (xx - px) ** 2 <= fit_radius**2
    for center in exclude_centers_yx or ():
        if center is None:
            continue
        cy, cx = map(float, center)
        fit_mask &= (yy - cy) ** 2 + (xx - cx) ** 2 > float(exclude_radius_px) ** 2

    model = np.zeros_like(cube, dtype=np.float64)
    amplitudes = np.full(nz, np.nan, dtype=np.float64)
    backgrounds = np.full(nz, np.nan, dtype=np.float64)
    n_fit = np.zeros(nz, dtype=np.int32)

    def _fit_range(z0, z1):
        # Per-channel work identical to the serial loop; disjoint output slots.
        for z in range(z0, z1):
            psf = psf_image((ny, nx), wave[z], primary_yx, psf_model)
            data = cube[z]
            valid = fit_mask & np.isfinite(data) & np.isfinite(psf)
            if variance is not None:
                valid &= np.isfinite(variance[z]) & (variance[z] > 0)
                weight = 1.0 / variance[z][valid]
            else:
                weight = np.ones(np.count_nonzero(valid), dtype=np.float64)
            if np.count_nonzero(valid) < 3:
                continue
            a = np.column_stack([psf[valid], np.ones(np.count_nonzero(valid), dtype=np.float64)])
            sw = np.sqrt(weight)
            try:
                coeff, *_ = np.linalg.lstsq(a * sw[:, None], data[valid] * sw, rcond=None)
            except np.linalg.LinAlgError:
                continue
            amp = float(coeff[0])
            bg = float(coeff[1])
            amplitudes[z] = amp
            backgrounds[z] = bg
            n_fit[z] = int(np.count_nonzero(valid))
            model[z] = amp * psf

    run_channel_chunks(_fit_range, nz, n_jobs=n_jobs)
    meta = {
        "amplitude_median": None if not np.any(np.isfinite(amplitudes)) else float(np.nanmedian(amplitudes)),
        "background_median": None if not np.any(np.isfinite(backgrounds)) else float(np.nanmedian(backgrounds)),
        "n_fit_median": int(np.nanmedian(n_fit)) if n_fit.size else 0,
        "fit_radius_px": float(fit_radius),
        "exclude_radius_px": float(exclude_radius_px),
    }
    return model, meta


## 5 · Chequeo de deriva


In [ ]:
import ast as _ast, hashlib as _hashlib

_SHAS = {
    "musepipe/stats.py:finite_values": "8aa861655f2b",
    "musepipe/stats.py:robust_sigma": "ef2aa72a72de",
    "musepipe/stats.py:robust_sigma_axis0": "0b976272de28",
    "musepipe/apertures.py:angular_separation_deg": "ce3b83206746",
    "musepipe/apertures.py:aperture_weights": "d8e1fd8bb88d",
    "musepipe/apertures.py:same_radius_control_positions": "fb89fcf9dd7d",
    "musepipe/extraction/aperture.py:_as_cube": "67ede036d305",
    "musepipe/extraction/aperture.py:annulus_background_spectrum": "68ec4c4e7299",
    "musepipe/extraction/aperture.py:_flag_window": "7ba686789970",
    "musepipe/extraction/aperture.py:channel_flags": "dbceb28a0a0b",
    "musepipe/extraction/aperture.py:aperture_correction_from_psf": "8ed04307abec",
    "musepipe/extraction/optimal.py:circular_window_indices": "ef251183f52a",
    "musepipe/extraction/optimal.py:normalized_psf_window": "9e755174b9e5",
    "musepipe/extraction/optimal.py:covariance_factor_for_npix": "bfff5c5c63d8",
    "musepipe/extraction/optimal.py:_channel_estimate": "e8b851d36243",
    "musepipe/extraction/optimal.py:estimate_variance_cube": "52dd9aee1ded",
    "musepipe/extraction/optimal.py:optimal_raw_spectrum": "ddb22d1bcebd",
    "musepipe/extraction/optimal.py:control_optimal_spectra": "f99f750cdfa0",
    "musepipe/extraction/optimal.py:psf_image": "c646ec012650",
    "musepipe/extraction/optimal.py:fit_primary_psf_model_cube": "b012c1e0484a"
}

def chequeo_de_deriva(shas=_SHAS, root=ROOT):
    problemas = []
    for key, sha in shas.items():
        rel, name = key.rsplit(':', 1)
        text = (root / rel).read_text(encoding='utf-8')
        lines = text.splitlines(keepends=True)
        node = next((n for n in _ast.parse(text).body
                     if isinstance(n, (_ast.FunctionDef, _ast.ClassDef)) and n.name == name),
                    None)
        if node is None:
            problemas.append(f'{key}: ya no existe en musepipe'); continue
        start = min([node.lineno] + [d.lineno for d in node.decorator_list]) - 1
        src = ''.join(lines[start:node.end_lineno]).rstrip('\n')
        actual = _hashlib.sha256(src.encode('utf-8')).hexdigest()[:12]
        if actual != sha:
            problemas.append(f'{key}: la copia es {sha}, musepipe tiene {actual}')
    return problemas

_deriva = chequeo_de_deriva()
if _deriva:
    print('DERIVA — la cadena cambió y esta copia se quedó atrás:')
    for p in _deriva:
        print('  ·', p)
    print(f'\nRegenera: python scripts/build_debug_notebooks.py --target {TARGET} C3')
else:
    print(f'sin deriva: las {len(_SHAS)} piezas copiadas son las de musepipe')


## 6 · El modelo de la primaria (lo que separa las dos variantes)

`psfsub` necesita restar la primaria antes de extraer. El ajuste es canal a canal, con la PSF de C1, **excluyendo un disco alrededor del compañero** para no absorberlo en el modelo de la estrella — si ese radio se queda corto, el modelo se come parte del compañero y `psfsub` sale bajo. Es una de las perillas interesantes de tocar.

*(Es la celda cara: ajusta un modelo por canal. Un par de minutos.)*


In [ ]:
primary_model, psfsub_meta = fit_primary_psf_model_cube(
    STAGE02, WAVE, STAR_YX, PSF_MODEL,
    variance_zyx=STAT_CUBE,
    fit_radius_px=PRIMARY_FIT_RADIUS,
    exclude_centers_yx=[OBJECT_YX],
    exclude_radius_px=PRIMARY_EXCL_RADIUS)
PSFSUB_CUBE = STAGE02 - primary_model
print('ajuste de la primaria:', {kk: psfsub_meta[kk] for kk in list(psfsub_meta)[:4]})

LAMBDA_VISTA = 7500.0    # canal que se dibuja; cámbialo y vuelve a ejecutar
iz = int(np.argmin(np.abs(WAVE - LAMBDA_VISTA)))
# El recorte tiene que contener a las DOS fuentes: centrado en la
# primaria con ±40 px dejaba al compañero (a ~70 px) fuera de cuadro,
# así que sus marcas no se veían y sus medianas salían NaN.
MARGEN_PX = 16.0
_ny, _nx = STAGE02.shape[1:]
sl = (slice(int(max(0, min(STAR_YX[0], OBJECT_YX[0]) - MARGEN_PX)),
            int(min(_ny, max(STAR_YX[0], OBJECT_YX[0]) + MARGEN_PX + 1))),
      slice(int(max(0, min(STAR_YX[1], OBJECT_YX[1]) - MARGEN_PX)),
            int(min(_nx, max(STAR_YX[1], OBJECT_YX[1]) + MARGEN_PX + 1))))
sy0, sx0 = sl[0].start, sl[1].start
dato, modelo_i = STAGE02[iz][sl], primary_model[iz][sl]
resid = PSFSUB_CUBE[iz][sl]

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
# Dato y modelo comparten escala LOG y los MISMOS límites: si cada uno
# llevara la suya, dos imágenes distintas se verían iguales.
pos = np.concatenate([dato[np.isfinite(dato) & (dato > 0)].ravel(),
                      modelo_i[np.isfinite(modelo_i) & (modelo_i > 0)].ravel()])
v_lo = float(np.nanpercentile(pos, 40)) if pos.size else 1e-3
v_hi = float(np.nanpercentile(pos, 99.99)) if pos.size else 1.0
norma = LogNorm(vmin=max(v_lo, 1e-6), vmax=max(v_hi, v_lo * 10))
for ax, img, titulo in ((axes[0], dato, 'B2 (con primaria)'),
                        (axes[1], modelo_i, 'modelo de la primaria')):
    im = ax.imshow(img, origin='lower', cmap='magma', norm=norma)
    ax.set_title(f'{titulo}  ·  LOG', fontsize=9)
    cb = fig.colorbar(im, ax=ax, fraction=0.046)
    cb.set_label(f'flujo [{UNIDAD}]', fontsize=7)
# El residuo es la resta de dos números grandes: se va a los dos signos
# y su interés está ALREDEDOR DE CERO. Con una escala secuencial
# anclada al máximo (el core de la estrella) todo lo demás sale negro,
# que es justo lo que no se quiere mirar. Va en divergente y simétrica,
# con los límites tomados FUERA del core: allí es donde el residuo
# importa, porque es el fondo sobre el que se mide el compañero.
yy_r, xx_r = np.indices(resid.shape, dtype=float)
r_core = np.hypot(yy_r - (STAR_YX[0] - sy0), xx_r - (STAR_YX[1] - sx0))
fuera = np.isfinite(resid) & (r_core > 6.0)
v = float(np.nanpercentile(np.abs(resid[fuera]), 98)) if fuera.any() else 1.0
im = axes[2].imshow(resid, origin='lower', cmap='RdBu_r', vmin=-v, vmax=v)
axes[2].set_title('residual = psfsub  ·  LINEAL simétrica', fontsize=9)
cb = fig.colorbar(im, ax=axes[2], fraction=0.046, extend='both')
cb.set_label(f'residuo [{UNIDAD}]', fontsize=7)
for ax in axes:
    ax.plot(STAR_YX[1] - sx0, STAR_YX[0] - sy0, '*', color='w', ms=11, mec='k')
    ax.plot(OBJECT_YX[1] - sx0, OBJECT_YX[0] - sy0, '+', color='tab:cyan', ms=9)
    ax.add_patch(Circle((OBJECT_YX[1] - sx0, OBJECT_YX[0] - sy0),
                        float(PRIMARY_EXCL_RADIUS), fill=False,
                        color='tab:cyan', lw=1.0, ls='--'))
    ax.add_patch(Circle((STAR_YX[1] - sx0, STAR_YX[0] - sy0),
                        float(PRIMARY_FIT_RADIUS), fill=False,
                        color='w', lw=0.9, ls=':'))
    # Los círculos se salen del recorte; sin fijar los límites estiran
    # los ejes y la imagen queda flotando en un marco blanco.
    ax.set_xlim(-0.5, resid.shape[1] - 0.5)
    ax.set_ylim(-0.5, resid.shape[0] - 0.5)
    ax.set_xlabel('x [px]')
axes[0].set_ylabel('y [px]')
fig.suptitle(f'λ = {WAVE[iz]:.0f} Å  ·  punteado blanco: radio de ajuste;'
             ' discontinuo cian: disco excluido del compañero', fontsize=9)
fig.tight_layout(); plt.show()

# Cuánto queda tras restar, donde se mide: si el residuo en la ventana
# no está centrado en cero, `psfsub` arrastra un pedestal.
en_ventana = (np.hypot(yy_r - (OBJECT_YX[0] - sy0), xx_r - (OBJECT_YX[1] - sx0))
              <= float(WINDOW_RADIUS_PX))
print(f'en la ventana del compañero (λ={WAVE[iz]:.0f} Å):')
print(f'  B2       mediana {float(np.nanmedian(dato[en_ventana])):9.3f} {UNIDAD}')
print(f'  modelo   mediana {float(np.nanmedian(modelo_i[en_ventana])):9.3f}')
print(f'  residual mediana {float(np.nanmedian(resid[en_ventana])):9.3f}'
      '   <- lo que psfsub deja bajo el compañero')
print(f'  residual mediana fuera del core (referencia): '
      f'{float(np.nanmedian(resid[fuera])):9.3f}')


## 7 · El peso óptimo, píxel a píxel

Aquí es donde C3 se diferencia de C2, y conviene verlo en un canal antes de lanzarlo sobre los 3681. La extracción óptima de Horne no suma: **promedia con pesos**,

> `f = Σ (P·D/V) / Σ (P²/V)`

con `D` el dato, `V` la varianza y `P` el **perfil de PSF normalizado** en la ventana (de C1, evaluado a esa λ). Cada píxel pesa `P/V`: mucho donde se espera señal y poco ruido, casi nada en el borde de la ventana. Por eso gana S/N frente a sumar una caja, donde todos los píxeles pesan igual y los del borde solo meten ruido.

Todo lo de esta sección es **del compañero**, no de la primaria: la ventana está centrada en `OBJECT_YX` y el cubo es el de la variante que elijas en `CUBO_VISTO` (`PSFSUB_CUBE`, ya con la primaria restada, o `LS_CUBE`). El título de la figura lo dice en cada ejecución.

Los cuatro mapas de abajo son, en la ventana y a esa λ: el **dato**, el **perfil `P`** que hace de peso, la **varianza** que entra como `1/V`, y la **contribución de cada píxel al flujo final**. Ese último es el que contesta «¿de dónde sale este número?».

### Por qué la contribución tiene valores positivos y negativos

Porque el **dato** los tiene. La contribución de un píxel es

> `cᵢ = (Pᵢ · Dᵢ / Vᵢ) / Σ(P²/V)`,  y  `Σᵢ cᵢ = f`

y como `Pᵢ > 0` y `Vᵢ > 0`, **el signo de `cᵢ` es el signo de `Dᵢ`**. En un canal suelto el compañero está por debajo del ruido, así que la mitad de los píxeles tienen el dato por debajo de cero (fluctuación, o resta de fondo/PSF que se pasó) y restan al total. Eso es lo normal y lo correcto: el estimador **no** es una suma de cosas positivas, es un promedio ponderado, y el flujo sale de la **cancelación** de ruido alrededor de un valor pequeño.

Lo que sí sería sospechoso es un patrón: si los negativos se agruparan en un lado (gradiente de fondo mal restado) o justo en el centro (sobre-sustracción de la primaria). Repartidos como sal y pimienta, es ruido.

Y en rojo, los píxeles que el **σ-clipping** descartó: el estimador itera `CLIP_SIGMA` veces quitando los que se desvían del modelo `f·P`. Es una protección contra cósmicos y píxeles malos, pero si el rechazo se concentra **sobre el compañero** se estaría recortando la señal y llamándolo ruido — es exactamente lo que vigila el chequeo `v4_clip_concentration` de la spec.

### Qué es el perfil radial de la última figura

El eje x es la **distancia de cada píxel al centro del compañero** (`OBJECT_YX`, el que fijó B3), en píxeles; el eje y es su flujo, con la barra `√V`. Los puntos son los píxeles de la ventana, sin ordenar ni promediar: cada uno es un píxel.

Lo que caracteriza es **si el dato tiene la forma que el estimador supone**. La línea azul es `f·P`: el modelo que resulta del ajuste, o sea *el mismo perfil de PSF, escalado al flujo que salió*. Si el dato siguiera una campana más ancha o más estrecha que esa línea, el peso estaría mal puesto y el estimador sería solo distinto, no óptimo (es lo que mide `psf_sensitivity` en el QC, moviendo la FWHM ±10 %). Y si hubiera una **pendiente** en los puntos que la línea no sigue, sería fondo sin restar dentro de la ventana.

> **Lo que se ve en el perfil radial no es una detección, y está bien.** En *un* canal el compañero queda por debajo de la barra de error: el modelo `f·P` sale casi plano. La señal aparece al juntar los 3681 canales, que es lo que hace la sección siguiente. Si en un solo canal se viera un pico limpio, habría que desconfiar.


In [ ]:
LAMBDA_PESO = LAMBDA_VISTA      # el mismo canal de la sección anterior
CUBO_VISTO = PSFSUB_CUBE        # prueba con LS_CUBE para ver la otra variante

ETIQUETA_CUBO = ('psfsub (primaria restada)' if CUBO_VISTO is PSFSUB_CUBE
                 else 'ls (cubo crudo + anillo)')
iz2 = int(np.argmin(np.abs(WAVE - LAMBDA_PESO)))
wy, wx = circular_window_indices(CUBO_VISTO.shape[1:], OBJECT_YX, WINDOW_RADIUS_PX)
P = normalized_psf_window(PSF_MODEL, WAVE[iz2], OBJECT_YX, wy, wx)
D = CUBO_VISTO[iz2][wy, wx]
if STAT_CUBE is not None:
    V = STAT_CUBE[iz2][wy, wx] * STAT_FACTOR
else:
    V = np.full(P.shape, float(robust_sigma(D)) ** 2)
valido = np.isfinite(D) & np.isfinite(V) & (V > 0)
f_ch, var_ch, neff, frac_clip, recortados = _channel_estimate(
    D, V, P, valido, clip_sigma=CLIP_SIGMA, clip_max_iter=CLIP_MAX_ITER)

usados = valido & ~recortados
r_px = np.hypot(wy - OBJECT_YX[0], wx - OBJECT_YX[1])
contrib = np.zeros_like(D)
denom = float(np.sum(P[usados] ** 2 / V[usados]))
contrib[usados] = P[usados] * D[usados] / V[usados] / denom   # suman f_ch
orden = np.argsort(P)[::-1]
top = orden[:max(1, P.size // 10)]
print(f'λ = {WAVE[iz2]:.0f} Å · ventana de {P.size} px'
      f' ({int(usados.sum())} usados, {int(recortados.sum())} recortados'
      f' = {100 * frac_clip:.1f}%)')
print(f'  f = {f_ch:.3f} ± {np.sqrt(var_ch):.3f} {UNIDAD}   (npix_eff = {neff:.1f})')
print(f'  píxeles que restan al total (dato < 0): '
      f'{int((contrib[usados] < 0).sum())} de {int(usados.sum())}'
      '  <- normal: en un canal el compañero está bajo el ruido')
print(f'  el 10% de píxeles con más peso aporta el'
      f' {100 * float(np.sum(contrib[top])) / f_ch:.0f}% del flujo'
      f'  <- esto es lo que una caja reparte por igual')
print(f'  suma simple de la ventana (sin pesos): {float(np.nansum(D[usados])):.3f}'
      '  (no es comparable: no lleva apcorr ni normalización)')

# El chequeo `v4_clip_concentration` de la spec, en este canal: si el
# rechazo se ceba en el núcleo del compañero, se está recortando la
# señal y llamándola ruido.
RADIO_NUCLEO_PX = 2.0
nucleo = valido & (r_px <= RADIO_NUCLEO_PX)
if nucleo.any() and valido.any():
    tasa_nucleo = float(recortados[nucleo].mean())
    tasa_total = float(recortados[valido].mean())
    razon = tasa_nucleo / tasa_total if tasa_total > 0 else 0.0
    print(f'  clipping: {100 * tasa_total:5.1f}% en la ventana,'
          f' {100 * tasa_nucleo:5.1f}% en el núcleo (r<={RADIO_NUCLEO_PX:g} px)'
          f' -> {razon:.1f}×')
    if razon > 2.0:
        print('    AVISO: el rechazo se concentra sobre el compañero.'
              ' Es lo que vigila v4_clip_concentration: el modelo f·P se'
              ' queda corto en el núcleo y sus píxeles salen como outliers.')

def _mapa(valores):
    """Los píxeles dispersos de la ventana, de vuelta a una imagen."""
    m = np.full((wy.max() - wy.min() + 1, wx.max() - wx.min() + 1), np.nan)
    m[wy - wy.min(), wx - wx.min()] = valores
    return m

paneles = ((_mapa(D), f'dato D [{UNIDAD}]', 'viridis', None),
           (_mapa(P), 'perfil P (normalizado, Σ=1)', 'magma', None),
           (_mapa(V), f'varianza V [{UNIDAD}²]', 'cividis', None),
           (_mapa(contrib), f'contribución al flujo [{UNIDAD}]', 'RdBu_r', 'sim'))
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (img, titulo, cmap, modo) in zip(axes, paneles):
    if modo == 'sim':
        v = float(np.nanpercentile(np.abs(img), 99.5))
        im = ax.imshow(img, origin='lower', cmap=cmap, vmin=-v, vmax=v)
    else:
        lo, hi = np.nanpercentile(img, [1, 99.5])
        im = ax.imshow(img, origin='lower', cmap=cmap, vmin=lo, vmax=hi)
    ax.set_title(titulo, fontsize=8)
    fig.colorbar(im, ax=ax, fraction=0.046)
    if recortados.any():
        ax.plot(wx[recortados] - wx.min(), wy[recortados] - wy.min(), 'x',
                color='tab:red', ms=5, mew=1.2)
    ax.plot(OBJECT_YX[1] - wx.min(), OBJECT_YX[0] - wy.min(), '+',
            color='w', ms=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle(f'ventana del COMPAÑERO sobre el cubo {ETIQUETA_CUBO}'
             f'  ·  λ = {WAVE[iz2]:.0f} Å  ·  × roja = recortado por el σ-clipping',
             fontsize=9)
fig.tight_layout(); plt.show()

# El perfil radial: si el modelo `f·P` no sigue al dato, el peso está
# mal puesto y el estimador no es óptimo, solo distinto.
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.errorbar(r_px[usados], D[usados], yerr=np.sqrt(V[usados]), fmt='o', ms=3,
            lw=0.6, color='0.4', alpha=0.8, label='dato ± √V')
if recortados.any():
    ax.plot(r_px[recortados], D[recortados], 'x', color='tab:red', ms=6,
            label='recortado por clipping')
ord_r = np.argsort(r_px)
ax.plot(r_px[ord_r], (f_ch * P)[ord_r], lw=1.6, color='tab:blue',
        label='modelo ajustado  f · P')
ax.axhline(0, color='0.7', lw=0.6)
ax.set_xlabel('distancia al centro del compañero [px]')
ax.set_ylabel(f'flujo [{UNIDAD}]')
ax.set_title(f'perfil radial del COMPAÑERO en su ventana · cubo {ETIQUETA_CUBO}'
             f' · λ = {WAVE[iz2]:.0f} Å', fontsize=9)
ax.legend(fontsize=8); fig.tight_layout(); plt.show()


## 8 · Las dos extracciones

El mismo estimador sobre los dos cubos. `optimal_raw_spectrum` devuelve además la varianza propagada, `npix_eff` y la fracción de píxeles rechazados por canal — el clipping es el que hay que vigilar: si se concentra en el compañero, se está recortando la señal (es el chequeo `v4_clip_concentration` de la spec).


In [ ]:
def extrae(cube, variance, etiqueta):
    # El peso del estimador es 1/varianza, y la varianza lleva el factor de M5.
    if variance is not None:
        variance = np.asarray(variance, dtype=float) * STAT_FACTOR
    else:
        variance = estimate_variance_cube(cube)
    bkg = None
    if LOCAL_BKG_ANNULUS_PX is not None:
        bkg = annulus_background_spectrum(
            cube, OBJECT_YX, LOCAL_BKG_ANNULUS_PX[0], LOCAL_BKG_ANNULUS_PX[1],
            exclude_yx=STAR_YX,
            exclude_radius=(LOCAL_BKG_ANNULUS_PX[2] if len(LOCAL_BKG_ANNULUS_PX) > 2 else 30.0))
    raw = optimal_raw_spectrum(cube, variance, WAVE, OBJECT_YX, PSF_MODEL,
                               window_radius_px=WINDOW_RADIUS_PX,
                               clip_sigma=CLIP_SIGMA, clip_max_iter=CLIP_MAX_ITER,
                               n_jobs=1, bkg_spectrum=bkg)
    cov = covariance_factor_for_npix(raw['npix_eff'], COV_FACTOR)
    raw_var = raw['variance'] * cov
    ctrl_yx, ctrl = control_optimal_spectra(
        cube, variance, WAVE, OBJECT_YX, STAR_YX, PSF_MODEL,
        window_radius_px=WINDOW_RADIUS_PX, clip_sigma=CLIP_SIGMA,
        clip_max_iter=CLIP_MAX_ITER, n_controls=N_CONTROLS,
        exclude_angle_deg=EXCLUDE_ANGLE_DEG, n_jobs=1,
        local_bkg_annulus_px=LOCAL_BKG_ANNULUS_PX)
    err_emp = (robust_sigma_axis0(ctrl) if ctrl.shape[0] >= 2
               else np.full(WAVE.size, robust_sigma(raw['flux'])))
    usable = (variance is not None and str(ERROR_MODE).lower() != 'empirical'
              and STAT_STATUS.lower() != 'red')
    err = np.sqrt(np.clip(raw_var, 0.0, np.inf)) if usable else np.asarray(err_emp, float)
    modo = 'stat' if usable else 'empirical'
    apert = {'kind': 'circle', 'radius_px': float(WINDOW_RADIUS_PX),
             'name': f'optimal_r{float(WINDOW_RADIUS_PX):g}'}
    apcorr, apcorr_mode, _nr = aperture_correction_from_psf(
        WAVE, apert, PSF_MODEL, center_yx=OBJECT_YX, correction_mode=APCORR_MODE)
    print(f'{etiqueta:8s} modo={modo:9s} apcorr={float(np.nanmedian(apcorr)):6.2f} '
          f'npix_eff={float(np.nanmedian(raw["npix_eff"])):6.1f} '
          f'clip_medio={100 * float(np.nanmedian(raw["clip_fraction"])):.2f}%')
    return {'raw': raw, 'flux': raw['flux'] * apcorr, 'err': err * apcorr,
            'err_emp': np.asarray(err_emp, float) * apcorr, 'apcorr': apcorr,
            'modo': modo, 'controles': ctrl, 'n_ctrl': len(ctrl_yx)}

LS     = extrae(LS_CUBE, STAT_CUBE, 'ls')
PSFSUB = extrae(PSFSUB_CUBE, STAT_CUBE, 'psfsub')


## 9 · Las dos variantes, una al lado de la otra

Es la comparación que D1 consume. Una diferencia **estructurada** entre ellas no es ruido: es el modelo de halo, porque el objeto y el estimador son los mismos y lo único que cambia es qué se restó antes.


In [ ]:
from musepipe.spectral import median_filter_1d
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                             gridspec_kw={'height_ratios': [2, 1]})
a1.plot(WAVE, median_filter_1d(LS['flux'], 41), lw=1.1, label='optimal_ls')
a1.plot(WAVE, median_filter_1d(PSFSUB['flux'], 41), lw=1.1, label='optimal_psfsub')
a1.axhline(0, color='0.7', lw=0.6); a1.axvline(6563, color='tab:red', ls=':')
a1.legend(fontsize=8); a1.set_ylabel('flujo (mediana 41 ch)')
a2.plot(WAVE, median_filter_1d(LS['flux'] - PSFSUB['flux'], 41), lw=1.0, color='tab:purple')
a2.axhline(0, color='0.7', lw=0.6)
a2.set_ylabel('ls − psfsub'); a2.set_xlabel('λ [Å]')
a1.set_title('las dos variantes: mismo estimador, distinto fondo', fontsize=9)
fig.tight_layout(); plt.show()
for nombre, v in (('ls', LS), ('psfsub', PSFSUB)):
    print(f"{nombre:8s} flujo mediano = {float(np.nanmedian(v['flux'])):9.2f}"
          f"  error mediano = {float(np.nanmedian(v['err'])):8.2f}  controles = {v['n_ctrl']}")


## 10 · El espectro binado con su error (los dos métodos)

El flujo por canal es demasiado ruidoso para leerse, y una mediana móvil suaviza pero **no dice cuánto vale lo que se ve**. Aquí se **bina**: se agrupan `BIN_CANALES` canales, el flujo es la media y el error se propaga como gaussiano independiente,

> σ_bin = √(Σ σᵢ²) / n

que para σ constante es el conocido σ/√n. Así cada punto lleva su barra y se puede juzgar si el espectro está por encima de cero — y, sobre todo, **comparar las dos variantes con una barra de error delante**, que es lo que decide si su diferencia es real o es ruido.

> **Aviso que la propia cadena mide**: esa fórmula supone canales **independientes**, y no lo son. G1 midió `n_eff/n ≈ 0.43` (el remuestreo en λ correlacionó canales vecinos), así que el error binado gaussiano está **subestimado en ~√(1/0.43) ≈ 1.5×**. Se dibujan las dos barras.


In [ ]:
BIN_CANALES = 25        # cámbialo y vuelve a ejecutar
N_EFF_OVER_N = 0.43     # medido por G1 (docs/noise_model.md)
ROJO_A = (7500.0, 9000.0)

def binea(wave, flujo, err, n):
    """Media por bloques de n canales, con error gaussiano independiente."""
    n = int(n)
    corte = (wave.size // n) * n
    w = wave[:corte].reshape(-1, n)
    f = np.asarray(flujo, dtype=float)[:corte].reshape(-1, n)
    e = np.asarray(err, dtype=float)[:corte].reshape(-1, n)
    bueno = np.isfinite(f) & np.isfinite(e)
    cuenta = bueno.sum(axis=1)
    with np.errstate(invalid='ignore', divide='ignore'):
        wb = np.nanmean(np.where(bueno, w, np.nan), axis=1)
        fb = np.nansum(np.where(bueno, f, 0.0), axis=1) / np.maximum(cuenta, 1)
        eb = np.sqrt(np.nansum(np.where(bueno, e, 0.0) ** 2, axis=1)) / np.maximum(cuenta, 1)
    vacio = cuenta == 0
    fb[vacio] = np.nan; eb[vacio] = np.nan
    return wb, fb, eb, cuenta

fig, ejes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ax, (nombre, v) in zip(ejes, (('optimal_ls', LS), ('optimal_psfsub', PSFSUB))):
    wb, fb, eb, _ = binea(WAVE, v['flux'], v['err_emp'], BIN_CANALES)
    eb_corr = eb / np.sqrt(N_EFF_OVER_N)   # canales correlacionados (G1)
    rojo_b = (wb >= ROJO_A[0]) & (wb <= ROJO_A[1])
    n_pts = int(np.isfinite(fb).sum())
    f_med = float(np.nanmedian(fb[rojo_b]))
    e_med = float(np.nanmedian(eb_corr[rojo_b]))
    snr_med = float(np.nanmedian(np.abs(fb[rojo_b]) / eb_corr[rojo_b]))
    print(f'{nombre:15s} {n_pts:3d} puntos de {BIN_CANALES} ch |'
          f' en {ROJO_A[0]:.0f}-{ROJO_A[1]:.0f} Å: flujo {f_med:9.1f}'
          f' ± {e_med:6.1f} -> S/N {snr_med:5.2f}')
    ax.plot(WAVE, v['flux'], lw=0.3, color='0.78', label='flujo por canal')
    ax.errorbar(wb, fb, yerr=eb_corr, fmt='o', ms=3, lw=0.9, color='tab:blue',
                ecolor='tab:blue', alpha=0.9,
                label=f'binado {BIN_CANALES} ch, ±σ corregido por n_eff')
    ax.errorbar(wb, fb, yerr=eb, fmt='none', lw=1.8, ecolor='tab:orange', alpha=0.8,
                label='±σ gaussiano (subestima: canales correlacionados)')
    ax.axhline(0, color='0.5', lw=0.7)
    ax.axvline(6562.8, color='tab:red', ls=':', label='Hα')
    fin_b = np.isfinite(fb)
    if fin_b.any():
        ax.set_ylim(*np.nanpercentile(fb[fin_b], [1, 99]) * np.array([2.5, 2.5]))
    ax.set_ylabel(f'{nombre}  [{UNIDAD}]', fontsize=8)
ejes[0].legend(fontsize=7, ncol=2)
ejes[-1].set_xlabel('λ [Å]')
ejes[0].set_title('las dos variantes del COMPAÑERO, binadas y con su error',
                  fontsize=9)
fig.tight_layout(); plt.show()


## 11 · Tres controles con geometría fija (los dos métodos)

Los `N_CONTROLS` controles de la extracción los reparte el pipeline en ángulo. Aquí se miran **tres sitios elegidos a mano**, a la misma distancia de la primaria que el compañero: el **opuesto** (PA + 180°) y los dos **perpendiculares** (PA ± 90°). Se colocan con los helpers de B3, así que heredan su convención de PA y el `north_angle_deg` del run.

Cada uno pasa por **exactamente el mismo proceso** que el compañero —misma ventana, mismo fondo, mismo estimador óptimo, misma `apcorr`— y para **las dos variantes**. Ahí no hay ninguna fuente, así que lo que se vea es halo residual y ruido.

Es la prueba directa de qué separa a `ls` de `psfsub`: **no es lo que hacen sobre el compañero, es lo que dejan donde no hay nada**. Si una deja los controles centrados en cero y la otra no, esa otra arrastra un pedestal que en el compañero no se distingue de flujo.


In [ ]:
from musepipe.stages.stage01c_localize import position_from_sep_pa
from musepipe.spectral import median_filter_1d

astro = qc_b3.get('astrometry') or {}
escala = qc_b3.get('pixel_scale_arcsec')
norte = (qc_b3.get('wcs_orientation') or {}).get('north_angle_deg', 0.0)
TRES = [('opuesto', 180.0), ('perpendicular +90', 90.0), ('perpendicular -90', -90.0)]

def extrae_en(cube, pos):
    """El mismo estimador de la sección 8, en otra posición."""
    var = (np.asarray(STAT_CUBE, dtype=float) * STAT_FACTOR
           if STAT_CUBE is not None else estimate_variance_cube(cube))
    bkg = None
    if LOCAL_BKG_ANNULUS_PX is not None:
        r_ex = LOCAL_BKG_ANNULUS_PX[2] if len(LOCAL_BKG_ANNULUS_PX) > 2 else 30.0
        bkg = annulus_background_spectrum(
            cube, pos, LOCAL_BKG_ANNULUS_PX[0], LOCAL_BKG_ANNULUS_PX[1],
            exclude_yx=STAR_YX, exclude_radius=r_ex)
    crudo = optimal_raw_spectrum(cube, var, WAVE, pos, PSF_MODEL,
                                 window_radius_px=WINDOW_RADIUS_PX,
                                 clip_sigma=CLIP_SIGMA, clip_max_iter=CLIP_MAX_ITER,
                                 n_jobs=1, bkg_spectrum=bkg)
    return crudo['flux']

sel_r = (WAVE >= 7500) & (WAVE <= 9000)
fig, ejes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
for ax, (nombre, v, cubo) in zip(ejes, (('optimal_ls', LS, LS_CUBE),
                                        ('optimal_psfsub', PSFSUB, PSFSUB_CUBE))):
    print(f'{nombre}:  (mediana en 7500-9000 Å)')
    med_obj = float(np.nanmedian(v['flux'][sel_r]))
    print(f'  {"compañero":22s} {med_obj:10.1f}')
    ax.plot(WAVE, median_filter_1d(v['flux'], 41), lw=1.4, color='tab:blue',
            label='compañero')
    for etiqueta, delta in TRES:
        pos = position_from_sep_pa((float(STAR_YX[0]), float(STAR_YX[1])),
                                   float(astro['sep_arcsec']),
                                   float(astro['pa_deg']) + delta,
                                   float(escala), north_angle_deg=float(norte or 0.0))
        f_c = extrae_en(cubo, pos) * v['apcorr']
        ax.plot(WAVE, median_filter_1d(f_c, 41), lw=0.9, alpha=0.8, label=etiqueta)
        print(f'  {etiqueta:22s} {float(np.nanmedian(f_c[sel_r])):10.1f}'
              f'    en y={pos[0]:.1f} x={pos[1]:.1f}')
    ax.axhline(0, color='0.5', lw=0.7)
    ax.set_ylabel(f'{nombre}  [{UNIDAD}]', fontsize=8)
    ax.legend(fontsize=7, ncol=4)
ejes[-1].set_xlabel('λ [Å]')
ejes[0].set_title('el compañero y tres posiciones sin fuente, procesadas igual'
                  '  (mediana móvil 41 ch)', fontsize=9)
fig.tight_layout(); plt.show()


## 12 · Comparación con la cadena

Cada variante contra **su** producto. Con las perillas por defecto deben salir idénticas; si tocas `WINDOW_RADIUS_PX` o el radio de exclusión de la primaria, aquí se ve exactamente cuánto se movió cada una.


In [ ]:
from musepipe.extraction.product import SpectrumProduct

def compara(nombre, mio, fichero, rtol=1e-9):
    ref = SpectrumProduct.read(SD / fichero)
    ok = True
    print(f'{nombre} vs {fichero}:')
    for clave, a, b in (('flujo', mio['flux'], np.asarray(ref.flux, float)),
                        ('error', mio['err'], np.asarray(ref.flux_err, float)),
                        ('apcorr', mio['apcorr'], np.asarray(ref.apcorr, float))):
        fin = np.isfinite(a) & np.isfinite(b)
        d = np.abs(a - b)[fin]
        ig = np.isclose(a[fin], b[fin], rtol=rtol, atol=0.0)
        print(f'   {clave:7s} idénticos {100 * ig.mean():6.2f}% de {fin.sum()} canales'
              f' | máx |Δ| = {d.max():.3e}')
        ok &= bool(ig.all())
    return ok, ref

ok_ls, ref_ls = compara('optimal_ls    ', LS, 'spec_optimal_object.fits')
ok_ps, ref_ps = compara('optimal_psfsub', PSFSUB, 'spec_optimal_psfsub_object.fits')
print()
print('IDÉNTICO: la copia reproduce la cadena.' if (ok_ls and ok_ps) else
      'DIFIERE — si has tocado una perilla, es lo esperado; si no, revisa el chequeo de deriva.')

fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)
for ax, (nombre, mio, ref) in zip(axes, (('optimal_ls', LS, ref_ls),
                                         ('optimal_psfsub', PSFSUB, ref_ps))):
    ax.plot(WAVE, median_filter_1d(np.asarray(ref.flux, float), 41), lw=1.6,
            color='0.6', label='cadena')
    ax.plot(WAVE, median_filter_1d(mio['flux'], 41), lw=1.0, ls='--',
            color='tab:blue', label='este notebook')
    ax.set_ylabel(nombre, fontsize=9); ax.legend(fontsize=8)
axes[-1].set_xlabel('λ [Å]')
fig.tight_layout(); plt.show()


## 13 · Figura de paper y tabla — las dos variantes

Las figuras anteriores son de diagnóstico. Ésta es la que se publica, y por eso cambia en tres cosas:

- **Sin binar**: cada canal con su σ. Binar es cómodo para leer un continuo, pero esconde justo lo que se quiere enseñar (o no enseñar): que en Hα no hay nada por encima del ruido **a la resolución del dato**.
- **Dos barras de error**: la **empírica** (dispersión de los controles procesados igual que el objeto) como banda, y la **propagada del STAT** como línea. Que se vean las dos es la forma honesta de enseñar que el STAT del cubo no es σ ([`docs/noise_model.md`](../docs/noise_model.md)).
- **Marcado completo**: las **bandas telúricas** sombreadas por especie (O₂ naranja, H₂O cian) con la **transmisión medida esa noche** en la tira de arriba, las **líneas de acreción** por familia (Balmer, He I, prohibidas, O I, Ca II, Paschen) y las **líneas de emisión de cielo** en gris discontinuo.

### Por qué bandas telúricas y no líneas telúricas

A la resolución de MUSE (FWHM ≈ 2.5 Å) las líneas individuales de O₂ y H₂O **no se resuelven**: dentro de un píxel espectral caen muchas. Marcar líneas sueltas daría una precisión que el dato no tiene, así que se marcan **bandas**. `molecfit` no está disponible aquí y, en estos datos, **no convergió** (A3 corrigió con la estrella telúrica estándar), pero de ahí quedó una **curva de transmisión medida** en la misma rejilla de λ: eso es más específico que cualquier lista de laboratorio y es lo que se dibuja. Catálogo y curva: [`musepipe/telluric_lines.py`](../musepipe/telluric_lines.py).

### Y sus datos, en columnas

La celda **escribe la tabla** además de la figura, en **ECSV** (el estándar portable de astropy): texto plano, con las unidades y la procedencia en la cabecera, que se lee con `Table.read(ruta)` sin configurar nada y se puede mandar por correo. Una figura sin sus datos no es un resultado citable.

> Aquí sale **dos veces**, una por variante, de los números recalculados en este notebook (no del producto de la cadena): si has tocado una perilla, la figura y la tabla la llevan. Ficheros con sufijo `_debug`, que no pisan nada.


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    ROOT_P = ROOT
    METHOD_P = 'optimal_ls'
    PRODUCT_P = 'recalculado en C3_optimal_debug (variante ls)'
    TARGET_P = str(TARGET).replace(' ', '') + '_debug'
    BUNIT_P = BUNIT or 'ADU'
    W_P, F_P = WAVE, LS['flux']
    E_P, E_ALT_P = LS['err_emp'], LS['err']
    EXTRA_P = {'flux_err_stat': LS['err'], 'apcorr': LS['apcorr'],
               'flags': channel_flags(WAVE, bad_windows_A=BAD_WINDOWS_A,
                                      skyline_windows_A=SKYLINE_WINDOWS_A,
                                      interpolated_windows_A=INTERPOLATED_WIN_A)}
    MODO_P = LS['modo']
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'optimal_ls, rehecha en el notebook (C3 debug)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c3_optimal_debug'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_ls' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_ls' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_ls' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


In [ ]:
try:
    import numpy as np
    import matplotlib.pyplot as plt
    from musepipe.paper_spectrum import (paper_spectrum_figure, pretty_flux_unit,
                                         spectrum_table_meta, write_spectrum_table)
    from musepipe.telluric_lines import measured_transmission
    ROOT_P = ROOT
    METHOD_P = 'optimal_psfsub'
    PRODUCT_P = 'recalculado en C3_optimal_debug (variante psfsub)'
    TARGET_P = str(TARGET).replace(' ', '') + '_debug'
    BUNIT_P = BUNIT or 'ADU'
    W_P, F_P = WAVE, PSFSUB['flux']
    E_P, E_ALT_P = PSFSUB['err_emp'], PSFSUB['err']
    EXTRA_P = {'flux_err_stat': PSFSUB['err'], 'apcorr': PSFSUB['apcorr'],
               'flags': channel_flags(WAVE, bad_windows_A=BAD_WINDOWS_A,
                                      skyline_windows_A=SKYLINE_WINDOWS_A,
                                      interpolated_windows_A=INTERPOLATED_WIN_A)}
    MODO_P = PSFSUB['modo']
    # Cuando la etapa eligió el error empírico, la columna `flux_err` ES
    # la empírica: dibujar las dos encima fingiría dos estimaciones
    # independientes donde solo hay una.
    if E_ALT_P is not None and np.allclose(E_ALT_P, E_P, equal_nan=True):
        E_ALT_P = None
        EXTRA_P.pop('flux_err_stat', None)
        print('las dos columnas de error coinciden (modo empírico):'
              ' una sola banda, y una sola columna en la tabla')
    # La transmisión telúrica MEDIDA de este run (A3). Si el objeto se
    # redujo en modo `cascade` no existe suelta: se marcan las bandas del
    # catálogo sin la profundidad de esa noche, y se dice.
    trans = measured_transmission(RUN_ID, project_root=ROOT_P)
    print('transmisión telúrica:', trans['source'] if trans else
          'no medida en este run — se marcan las bandas del catálogo')
    # Los canales que la etapa marcó como malos (hueco del láser AO) no
    # se dibujan: valen 0, y un 0 pintado se lee como una medida.
    from musepipe.extraction.aperture import FLAG_BAD_WINDOW
    MALOS_P = (np.asarray(EXTRA_P.get('flags', 0), dtype=int) & FLAG_BAD_WINDOW) != 0
    fig, _ejes = paper_spectrum_figure(
        W_P, F_P, E_P, flux_err_alt=E_ALT_P, bad_channels=MALOS_P,
        err_label='±1σ empírico (controles procesados igual)', err_alt_label='±1σ propagado del STAT (no es σ)',
        transmission=trans,
        title=nb.display_name(RUN_ID) + ' · ' + 'optimal_psfsub, rehecha en el notebook (C3 debug)',
        flux_label='flujo [' + pretty_flux_unit(BUNIT_P) + ']')
    outdir = nb.run_dir(RUN_ID) / 'plots' / 'c3_optimal_debug'
    outdir.mkdir(parents=True, exist_ok=True)
    # PDF además de PNG: es la que va al paper, y en vectorial las
    # etiquetas de las 24 líneas siguen leyéndose al ampliar. El PNG a
    # 300 dpi es el mínimo que piden las revistas para figuras de línea.
    DPI_P = 300      # súbelo si necesitas más resolución
    for ext in ('png', 'pdf'):
        fig.savefig(outdir / ('spectrum_paper_psfsub' + '.' + ext), dpi=DPI_P)
    tabla = write_spectrum_table(
        nb.run_dir(RUN_ID) / 'tables' / ('spec_' + METHOD_P + '_' + TARGET_P + '.ecsv'),
        W_P, F_P, E_P, extra_columns=EXTRA_P,
        units={'flux': BUNIT_P, 'flux_err': BUNIT_P, 'flux_err_stat': BUNIT_P},
        meta=spectrum_table_meta(run_id=RUN_ID, target=TARGET_P, method=METHOD_P,
                                 product=PRODUCT_P, flux_unit=BUNIT_P,
                                 error_mode=MODO_P,
                                 extra={'figure': str(outdir / ('spectrum_paper_psfsub' + '.pdf'))}))
    print('figura ->', outdir / ('spectrum_paper_psfsub' + '.pdf'))
    print('tabla  ->', tabla, '(' + str(tabla.stat().st_size // 1024) + ' kB, '
          + str(int(np.size(W_P))) + ' canales)')
    print('        se lee con:  from astropy.table import Table; Table.read(ruta)')
    plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)
